#Research Paper Intelligence System

This notebook converts the uploaded `.py` file into a proper Google Colab notebook and repairs the main syntax/runtime issues.

## What it does
- Loads the `CShorten/ML-ArXiv-Papers` dataset
- Builds semantic embeddings with Sentence Transformers
- Searches papers using FAISSS
- Summarizes abstracts with BART
- Extracts keywords with KeyBERT
- Runs a named-entity-recognition (NER) example
- Offers optional Groq-powered paper comparison

## How to use
Run cells from top to bottom. The first run downloads models and data and may take a while. The notebook uses the first 15,000 records by default; reduce `MAX_PAPERS` if your session runs out of RAM.

**Important:** Colab's temporary storage is cleared when its runtime resets. Download or save important outputs if needed.

## 1. Install dependencies

In [1]:
%pip -q install datasets sentence-transformers faiss-cpu transformers keybert langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.4 MB/s eta 0:00:00


## 2. Load and prepare the dataset

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import os
import faiss

MAX_PAPERS = 15000  # Lower this (e.g. 2000) if RAM is limited.

dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
df = pd.DataFrame(dataset)

required_columns = {"title", "abstract"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df = df[["title", "abstract"]].dropna().copy()
df["title"] = df["title"].astype(str)
df["abstract"] = df["abstract"].astype(str)
df = df.head(MAX_PAPERS).reset_index(drop=True)

df["paper_text"] = (
    df["title"].str.replace("\\n", " ", regex=False).str.strip()
    + " "
    + df["abstract"].str.replace("\\n", " ", regex=False).str.strip()
)

print(f"Loaded {len(df):,} papers")
display(df[["title", "abstract"]].head())

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv: reconstructing file:   0%|          |  0.00B /  147MB            

ML-Arxiv-Papers.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

Loaded 15,000 papers


,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


## 3. Create sentence embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

EMBEDDINGS_PATH = "/content/paper_embeddings.npy"
embeddings = None

# Reuse a saved file only when its row count matches the current dataset.
if os.path.exists(EMBEDDINGS_PATH):
    saved = np.load(EMBEDDINGS_PATH)
    if saved.shape[0] == len(df):
        embeddings = saved
        print("Loaded cached embeddings:", embeddings.shape)
    else:
        print("Cached embeddings do not match this dataset size; regenerating.")

if embeddings is None:
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
    np.save(EMBEDDINGS_PATH, embeddings)
    print("Generated and saved embeddings:", embeddings.shape)

print("Embedding matrix:", embeddings.shape, embeddings.dtype)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

## 4. Build the FAISS similarity-search index

In [ ]:
# Embeddings are already L2-normalized, so inner product is cosine similarity.
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(np.ascontiguousarray(embeddings, dtype=np.float32))

print(f"FAISS index contains {index.ntotal:,} papers.")

## 5. Search for relevant papers

In [ ]:
def search_papers(query: str, k: int = 5):
    """Return the top-k papers most semantically similar to a query."""
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Please provide a non-empty search query.")
    k = max(1, min(int(k), index.ntotal))

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = index.search(
        np.ascontiguousarray(query_embedding), k
    )

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        results.append({
            "similarity": float(score),
            "title": df.iloc[int(idx)]["title"],
            "abstract": df.iloc[int(idx)]["abstract"],
            "index": int(idx),
        })
    return results

def print_search_results(query: str, k: int = 5):
    results = search_papers(query, k)
    for rank, paper in enumerate(results, start=1):
        print("=" * 90)
        print(f"Rank: {rank} | Similarity: {paper['similarity']:.4f}")
        print("Title:", paper["title"])
        print("\nAbstract:")
        print(paper["abstract"][:700])
        print()
    return results

results = print_search_results(
    "deep learning for medical image analysis",
    k=5,
)

## 6. Load BART and summarize abstracts

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=-1,  # CPU; change to 0 to use a GPU runtime.
)

def summarize_text(text: str, max_length: int = 120, min_length: int = 35) -> str:
    text = str(text).strip()
    if not text:
        return "No text available to summarize."
    # BART has a limited input length; truncate explicitly for predictable behavior.
    tokens = summarizer.tokenizer(
        text,
        truncation=True,
        max_length=1024,
        return_tensors=None,
    )
    prepared_text = summarizer.tokenizer.decode(
        tokens["input_ids"], skip_special_tokens=True
    )
    input_token_count = len(tokens["input_ids"])
    effective_max = min(max_length, max(20, input_token_count - 2))
    effective_min = min(min_length, max(5, effective_max - 5))
    output = summarizer(
        prepared_text,
        max_length=effective_max,
        min_length=effective_min,
        do_sample=False,
        truncation=True,
    )
    return output[0]["summary_text"]

print(summarize_text(df.iloc[0]["abstract"]))

## 7. Search and summarize papers

In [ ]:
def search_and_summarize(query: str, k: int = 5):
    papers = search_papers(query, k)
    for rank, paper in enumerate(papers, start=1):
        print("=" * 90)
        print(f"Rank: {rank}")
        print(f"Similarity: {paper['similarity']:.4f}")
        print("Title:", paper["title"])
        print("\nSummary:")
        print(summarize_text(paper["abstract"]))
        print()
    return papers

papers = search_and_summarize("Deep Learning in Medical Imaging", k=5)

## 8. Extract keywords with KeyBERT

In [ ]:
from keybert import KeyBERT

kw_model = KeyBERT(model=model)

def extract_keywords(text: str, top_n: int = 5):
    return kw_model.extract_keywords(
        str(text),
        keyphrase_ngram_range=(1, 3),
        stop_words="english",
        top_n=top_n,
    )

sample_abstract = df.iloc[0]["abstract"]
print(extract_keywords(sample_abstract, top_n=5))

## 9. Search, summarize, and show keywords

In [ ]:
def search_summarize_keywords(query: str, k: int = 5, top_n: int = 5):
    papers = search_papers(query, k)
    for rank, paper in enumerate(papers, start=1):
        print("=" * 90)
        print(f"Rank: {rank} | Similarity: {paper['similarity']:.4f}")
        print("Title:", paper["title"])
        print("\nSummary:")
        print(summarize_text(paper["abstract"]))
        print("\nKeywords:")
        for keyword, score in extract_keywords(paper["abstract"], top_n=top_n):
            print(f"- {keyword} ({score:.3f})")
        print()
    return papers

# Example:
# search_summarize_keywords("Vision Transformer for medical imaging", k=3)

## 10. Named Entity Recognition (NER) example

In [ ]:
ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=-1,
)

example_text = "ResNet50 was trained on ImageNet using PyTorch at Stanford University."
entities = ner(example_text)
display(pd.DataFrame(entities))

## 11. Optional: Groq-powered comparison of two papers

This section requires a Groq API key. In Colab, add a secret named `GROQ_API_KEY` using the **Secrets (key icon)** panel and enable notebook access for it. Do not paste your API key into a public notebook.

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if GROQ_API_KEY:
    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        api_key=GROQ_API_KEY,
        temperature=0,
    )
    print("Groq model is ready.")
else:
    llm = None
    print("Groq key not found. Add GROQ_API_KEY in Colab Secrets to enable this optional section.")

In [ ]:
def compare_papers(paper_query_1: str, paper_query_2: str) -> str:
    """Find two papers and ask the configured Groq model to compare them."""
    if llm is None:
        raise RuntimeError("Groq is not configured. Add GROQ_API_KEY in Colab Secrets first.")

    first = search_papers(paper_query_1, k=1)[0]
    second = search_papers(paper_query_2, k=1)[0]

    prompt = f"""
Compare these two research papers using only the supplied titles and abstracts.

Compare:
1. Research objective
2. Methodology
3. Key contributions
4. Advantages
5. Limitations
6. Possible applications

Paper 1
Title: {first['title']}
Abstract: {first['abstract']}

Paper 2
Title: {second['title']}
Abstract: {second['abstract']}

If the abstracts do not provide enough evidence for a point, say so. Present the result clearly with headings.
"""
    response = llm.invoke(prompt)
    return response.content

# Example after configuring GROQ_API_KEY:
# print(compare_papers("Vision Transformer", "CNN for medical image analysis"))

## Notes and troubleshooting
- If Colab runs out of RAM, set `MAX_PAPERS = 2000` in Section 2 and rerun from the top.
- If you change `MAX_PAPERS`, rerun the embedding and index cells. The notebook checks the cached embedding row count.
- If a model download fails, rerun the affected cell after the connection is stable.
- The Groq section is optional; the core semantic search, summarization, and keyword extraction do not require a Groq key.
- This notebook intentionally replaces the malformed/incomplete LangChain agent definitions in the original script with direct, testable functions and an optional Groq comparison function.